In [ ]:
import os
import subprocess
import json
import csv
import ast


REPOS = [
    "django/django",
    "pallets/flask",
    "tiangolo/fastapi",
    "tornadoweb/tornado",
    "PyCQA/bandit",
    "sqlmapproject/sqlmap",
    "andresriancho/w3af",

    "volatilityfoundation/volatility",
    "ansible/ansible",
    "apache/airflow",
    "saltstack/salt",
    "jupyter/notebook",
    "mlflow/mlflow",
    "psf/requests",
    "urllib3/urllib3",
    "paramiko/paramiko",
    "yaml/pyyaml",
]

CLONE_BASE  = "python_repos"
RULES_PATH  = "new_25_generated.yaml"
OUTPUT_CSV  = "python_data_w25.csv"


#function_extracting

def extract_enclosing_function_source(file_path: str, target_line: int) -> str:

    if not os.path.isfile(file_path) or not file_path.lower().endswith(".py"):
        return ""

    try:
        source = open(file_path, encoding="utf-8", errors="ignore").read()
    except Exception:
        return ""

    try:
        tree = ast.parse(source, filename=file_path)
    except SyntaxError:
        return ""

    candidate = None
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and hasattr(node, "end_lineno"):
            start, end = node.lineno, node.end_lineno
            if start <= target_line <= end:
                span = end - start
                if candidate is None or span < candidate[0]:
                    candidate = (span, start, end)

    if not candidate:
        return ""

    _, fn_start, fn_end = candidate
    lines = source.splitlines()
    return "\n".join(lines[fn_start - 1 : fn_end]).rstrip()

#line_extracting
def extract_simple_lines(file_path: str, start_line: int, end_line: int) -> str:

    if not os.path.isfile(file_path):
        return ""
    try:
        with open(file_path, encoding="utf-8", errors="ignore") as f:
            lines = f.read().splitlines()
    except Exception:
        return ""
    lo = max(0, start_line - 1)
    hi = min(len(lines), end_line)
    if lo >= hi:
        return ""
    return "\n".join(lines[lo:hi]).rstrip()



def run_semgrep(target_dir: str) -> dict:

    cmd = [
        "semgrep",
        "--json",
        "--no-git-ignore",
        f"--config={RULES_PATH}",
        "--config=p/python",
        "--timeout", "0",
        target_dir,
    ]
    print("Running Semgrep:", " ".join(cmd))
    # ADDED: encoding='utf-8'
    proc = subprocess.run(cmd, capture_output=True, text=True, check=False, encoding='utf-8')

    try:
        data = json.loads(proc.stdout)
    except json.JSONDecodeError:
        print("Semgrep JSON parse failed:")
        print(proc.stderr.strip())
        return {"results": []}

    if data.get("errors"):
        for e in data["errors"]:
            print("Semgrep error:", e.get("message", e))
        return {"results": []}

    return data



GIT_EXEC_PATH = r"C:\Program Files\Git\bin\git.exe"


if __name__ == "__main__":
    os.makedirs(CLONE_BASE, exist_ok=True)
    all_findings = []




    for repo in REPOS:
        print(f"\n=== Processing {repo} ===")
        owner, name = repo.split("/")
        repo_dir = os.path.join(CLONE_BASE, name)

        # Clone or pull with error handling
        if os.path.isdir(repo_dir):
            try:
                print(f"  Attempting to update {repo} (git pull) using '{GIT_EXEC_PATH}'...")
                # ADDED: encoding='utf-8'
                subprocess.run([GIT_EXEC_PATH, "-C", repo_dir, "pull"], check=True, capture_output=True, text=True, encoding='utf-8')
                print(f"  Successfully updated {repo}.")
            except subprocess.CalledProcessError as e:
                print(f"  Error updating {repo}: {e}")
                print(f"  Stdout: {e.stdout.strip()}")
                print(f"  Stderr: {e.stderr.strip()}")
                print(f"  Skipping analysis for {repo} due to update failure.")
                continue # Skip this repo if update fails
            except FileNotFoundError:
                print(f"  Error: Git executable not found at '{GIT_EXEC_PATH}'. Please ensure the path is correct or Git is in your system PATH.")
                print(f"  Skipping analysis for {repo}.")
                continue
            except Exception as e:
                print(f"  An unexpected error occurred while updating {repo}: {e}")
                print(f"  Skipping analysis for {repo} due to unexpected update failure.")
                continue # Skip this repo if an unexpected error occurs
        else:
            try:
                print(f"  Attempting to clone {repo} from GitHub using '{GIT_EXEC_PATH}'...")

                subprocess.run([GIT_EXEC_PATH, "clone", f"https://github.com/{repo}.git", repo_dir], check=True, capture_output=True, text=True, encoding='utf-8')
                print(f"  Successfully cloned {repo}.")
            except subprocess.CalledProcessError as e:
                print(f"  Error cloning {repo}: {e}")
                print(f"  Stdout: {e.stdout.strip()}")
                print(f"  Stderr: {e.stderr.strip()}")
                print(f"  Skipping analysis for {repo} due to clone failure.")
                continue # Skip this repo if clone fails
            except FileNotFoundError:
                print(f"  Error: Git executable not found at '{GIT_EXEC_PATH}'. Please ensure the path is correct or Git is in your system PATH.")
                print(f"  Skipping analysis for {repo}.")
                continue
            except Exception as e:
                print(f"  An unexpected error occurred while cloning {repo}: {e}")
                print(f"  Skipping analysis for {repo} due to unexpected clone failure.")
                continue # Skip this repo if an unexpected error occurs

        # Continue with Semgrep analysis only if clone/pull was successful
        sem_data = run_semgrep(repo_dir)
        results = sem_data.get("results", [])
        print(f" → {len(results)} findings in {repo}\n")

        for f in results:
            rpt = f["path"]
            sl  = f["start"]["line"]
            el  = f["end"]["line"]
            ex  = f.get("extra", {})
            meta = ex.get("metadata", {})


            full_path = None


            if os.path.isabs(rpt) and os.path.isfile(rpt):
                full_path = rpt
            else:

                norm_rpt = rpt.replace("\\", "/")
                prefix = repo_dir.replace("\\", "/") + "/"
                if norm_rpt.startswith(prefix):
                    stripped = norm_rpt[len(prefix):]
                    candidate = os.path.join(repo_dir, stripped)
                    if os.path.isfile(candidate):
                        full_path = candidate


            if full_path is None:
                candidate = os.path.join(repo_dir, rpt)
                if os.path.isfile(candidate):
                    full_path = candidate

            if full_path is None or not os.path.isfile(full_path):
                print(f"  File not found: {rpt} → attempted under {repo_dir}")
                snippet = ""
            else:
                #  If it’s a .py file, try AST extraction (micro-granularity)
                snippet = extract_enclosing_function_source(full_path, sl)
                #  If AST extraction failed or it’s not .py, fallback to raw lines
                if not snippet:
                    snippet = extract_simple_lines(full_path, sl, el)


            cwe_list = meta.get("cwe", [])
            if isinstance(cwe_list, str) and cwe_list.startswith("CWE-"):
                cwe_id = cwe_list
            elif isinstance(cwe_list, list):
                cwe_id = next((c for c in cwe_list if c.startswith("CWE-")), "CWE-Unknown")

            else:
                cwe_id = "CWE-Unknown"

            all_findings.append({
                "repo":       repo,
                "file":       os.path.relpath(full_path or "", repo_dir),
                "start_line": sl,
                "end_line":   el,
                "rule_id":    f.get("check_id", ""),
                "severity":   ex.get("severity", ""),
                "message":    ex.get("message", "").replace("\n", " "),
                "cwe_id":     cwe_id,
                "code_snip":  snippet
            })

    # to_csv

    fieldnames = [
        "repo", "file", "start_line", "end_line",
        "rule_id", "severity", "message", "cwe_id", "code_snip"
    ]
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as csvf:
        writer = csv.DictWriter(csvf, fieldnames=fieldnames, quoting=csv.QUOTE_ALL)
        writer.writeheader()
        for row in all_findings:
            writer.writerow(row)

    print(f"\n Done. Wrote {len(all_findings)} entries to {OUTPUT_CSV}")